# 1. Purpose of this notebook

# 2. Imports and DuckDB setup

In [2]:
# -----------------------------------------
# Imports
# -----------------------------------------

import duckdb
import pandas as pd
from pathlib import Path


# -----------------------------------------
# Project Paths
# -----------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "Data"
PROCESSED_PATH = DATA_PATH / "Processed"
SQL_OUTPUT_PATH = PROCESSED_PATH / "sql_outputs"

SQL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Processed data path:", PROCESSED_PATH)
print("SQL output path:", SQL_OUTPUT_PATH)

Processed data path: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\Data\Processed
SQL output path: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\Data\Processed\sql_outputs


In [3]:
# -----------------------------------------
# Convert curated dimension tables to Parquet
# -----------------------------------------

dimension_tables = [
    "dim_airlines",
    "dim_airports",
    "dim_cancellation_codes"
]

for table in dimension_tables:
    
    csv_path = PROCESSED_PATH / f"{table}.csv"
    parquet_path = PROCESSED_PATH / f"{table}.parquet"
    
    df = pd.read_csv(csv_path)
    df.to_parquet(parquet_path, index=False)
    
    print(f"Converted {table}.csv → {table}.parquet")

Converted dim_airlines.csv → dim_airlines.parquet
Converted dim_airports.csv → dim_airports.parquet
Converted dim_cancellation_codes.csv → dim_cancellation_codes.parquet


In [4]:
# -----------------------------------------
# Connect DuckDB
# -----------------------------------------

con = duckdb.connect()

# -----------------------------------------
# Register curated Parquet datasets as views
# -----------------------------------------

con.execute(f"""
CREATE OR REPLACE VIEW fact_flights AS
SELECT * FROM read_parquet('{(PROCESSED_PATH / "fact_flights.parquet").as_posix()}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW dim_airlines AS
SELECT * FROM read_parquet('{(PROCESSED_PATH / "dim_airlines.parquet").as_posix()}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW dim_airports AS
SELECT * FROM read_parquet('{(PROCESSED_PATH / "dim_airports.parquet").as_posix()}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW dim_cancellation_codes AS
SELECT * FROM read_parquet('{(PROCESSED_PATH / "dim_cancellation_codes.parquet").as_posix()}')
""")

print("DuckDB connected and views created.")

DuckDB connected and views created.


In [5]:
# -----------------------------------------
# Validate row counts
# -----------------------------------------

print("fact_flights:", con.execute("SELECT COUNT(*) FROM fact_flights").fetchone()[0])
print("dim_airlines:", con.execute("SELECT COUNT(*) FROM dim_airlines").fetchone()[0])
print("dim_airports:", con.execute("SELECT COUNT(*) FROM dim_airports").fetchone()[0])
print("dim_cancellation_codes:", con.execute("SELECT COUNT(*) FROM dim_cancellation_codes").fetchone()[0])

fact_flights: 5819079
dim_airlines: 14
dim_airports: 322
dim_cancellation_codes: 4


In [6]:
# Preview fact table columns
con.execute("DESCRIBE fact_flights").df()

,column_name,column_type,null,key,default,extra
0,flight_date,DATE,YES,None,None,None
1,AIRLINE,VARCHAR,YES,None,None,None
2,FLIGHT_NUMBER,INTEGER,YES,None,None,None
3,ORIGIN_AIRPORT,VARCHAR,YES,None,None,None
4,DESTINATION_AIRPORT,VARCHAR,YES,None,None,None
5,SCHEDULED_DEPARTURE,INTEGER,YES,None,None,None
6,TAIL_NUMBER,VARCHAR,YES,None,None,None
7,scheduled_departure_hour,TINYINT,YES,None,None,None
8,time_band,VARCHAR,YES,None,None,None
9,day_of_week,VARCHAR,YES,None,None,None


# 3. Define project paths

# 4. Convert curated dimensions to Parquet (one-time standardisation)

# 5. Connect DuckDB and register curated datasets


# 6. Validate row counts and schemas


# 7. Create analytical SQL views


# 8. Materialise selected analytical tables


# 9. Validate analytical outputs

# 10. Export final tables to Data/Processed